In [1]:
%pip install python-dotenv requests


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
from dotenv import load_dotenv

load_dotenv()
KEY: str = os.environ["KOREAN_DICT_KEY"]
print(KEY[:4] + "***")

0B2F***


Q1 우리말샘에서 단어 검색하기

In [10]:
import requests

def search_word(q: str, num: int = 10, start: int = 1) -> dict:
    """우리말샘 API에서 단어를 검색하고 JSON 응답을 반환한다."""
    url = "https://opendict.korean.go.kr/api/search"
    params = {
        "key": KEY,
        "q": q,
        "req_type": "json",
        "num": num,
        "start": start,
        "type1": "word",
    }
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    return r.json()

requests.get()으로 우리말샘 API에 HTTP GET 요청을 보내는 함수다. params 딕셔너리에 필요한 쿼리 매개변수를 담아 전달하면 requests가 자동으로 URL에 붙여준다. raise_for_status()는 404나 500 같은 오류 응답이 오면 예외를 발생시켜 문제를 즉시 알 수 있게 해준다.

In [ ]:
import json

data = search_word("김치")
print(json.dumps(data, ensure_ascii=False, indent=2)[:400])

{
  "channel": {
    "total": 328,
    "num": 10,
    "title": "우리말샘 개발 지원(Open API) - 사전 어휘 검색",
    "start": 1,
    "description": "우리말샘 개발 지원(Open API) - 사전 어휘 검색 결과",
    "link": "https://opendict.korean.go.kr",
    "item": [
      {
        "word": "김치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "",
          


json.dumps()는 파이썬 딕셔너리를 보기 좋은 JSON 문자열로 변환한다. ensure_ascii=False를 빼면 한글이 \uae40\uce58 같은 유니코드 이스케이프 시퀀스로 출력되어 사람이 읽기 어렵다. [:400]은 출력이 너무 길어지지 않도록 앞 400자만 자르는 것이다.

In [16]:
# (i) 전체 결과 수 및 이 페이지 항목 수
channel = data["channel"]
total = channel["total"]
items = channel["item"]
n = len(items)
print(f"총 {total}건, 이 페이지 {n}건")

# (ii) 첫 5개 항목 출력
for item in items[:5]:
    word = item.get("word", "")
    # pos는 sense 리스트 첫 번째 항목 안에 있음
    senses = item.get("sense", [])
    if senses:
        pos = senses[0].get("pos", "품사 없음") or "품사 없음"
        definition = senses[0].get("definition", "")
    else:
        pos = "품사 없음"
        definition = ""
    print(f"{word} ({pos}) --> {definition[:40]}")

총 328건, 이 페이지 10건
김치 (명사) --> 소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린
김-치 (명사) --> 고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 
김-치 (명사) --> 조선 중기의 문신(1577~1625). 자는 사정(士精). 호는 남봉(南
김치 공장 (품사 없음) --> 김치를 만드는 공장.
김치 보릿고개 (품사 없음) --> 김장철인 가을·겨울과 달리 상대적으로 김치가 부족한 봄여름을 비유적으로 


data["channel"]에 핵심 데이터가 담겨 있으며 total은 전체 검색 건수, item은 현재 페이지의 항목 리스트다. dict.get()을 사용하면 키가 없어도 기본값을 반환하므로 KeyError 없이 안전하게 접근할 수 있다. 품사(pos)와 뜻풀이(definition)는 sense 리스트의 첫 번째 요소 안에 위치한다.

Q2 여러 검색어로 비교하기

In [18]:
import time

words: list[str] = [
    "김치", "라면", "만두", "김밥",
    "국수", "떡볶이", "불고기", "비빔밥",
]

results: dict[str, dict] = {}

for w in words:
    d = search_word(w)
    results[w] = d
    total = d["channel"]["total"]
    print(f"{w}: {total}건")
    time.sleep(0.3)

김치: 328건
라면: 86건
만두: 89건
김밥: 39건
국수: 227건
떡볶이: 24건
불고기: 38건
비빔밥: 38건


8개 검색어를 for문으로 순서대로 반복하며 각각 search_word()로 API를 호출한다. results 딕셔너리에 검색어를 키로 응답 전체를 저장해두면 Q2(b)에서 재사용할 수 있다. time.sleep(0.3)으로 매 호출 사이에 0.3초 대기를 넣어 서버에 과도한 요청을 보내지 않도록 한다.

In [19]:
from collections import Counter

all_items: list[dict] = []
for d in results.values():
    all_items.extend(d["channel"].get("item", []))

pos_counter: Counter = Counter()
for item in all_items:
    senses = item.get("sense", [])
    pos = (senses[0].get("pos") if senses else None) or "(미상)"
    pos_counter[pos] += 1

print("품사 빈도 상위 3개:")
for pos, count in pos_counter.most_common(3):
    print(f"  {pos}: {count}건")

품사 빈도 상위 3개:
  명사: 60건
  (미상): 19건
  어미: 1건


 8개 검색어에서 받은 모든 항목을 all_items 하나의 리스트로 합친다. pos 키가 없거나 빈 문자열인 경우 모두 "(미상)"으로 처리한다. Counter.most_common(3)은 빈도가 높은 순으로 상위 3개를 자동 정렬해 반환한다.
관찰: 가장 흔한 품사는 명사일 가능성이 높다. 음식 관련 단어는 대부분 사물이나 개념을 지칭하는 명사이며, 파생어와 합성어도 대부분 명사로 등록되기 때문이다.